# 03 · Gold · `receita_decomposicao`
### V-Energy Grid 360 — Arquitetura Medalhão

Responde à pergunta central do case, a do Henrique: *"quero ver por bairro e por classe, e quero saber quanto da variação foi preço e quanto foi quantidade de energia. Se foi volume, quero saber onde e por quê."* Grava `v_energy.gold.receita_decomposicao`, seguindo o **contrato da Gold (tabela A4, versão v4)**, com as 24 colunas do contrato.

**Grão:** 1 linha = 1 combinação **município × bairro × classe**, comparando o **primeiro ano (período 0: `2022-07` a `2023-06`)** com os **últimos 12 meses (período 1: `2025-07` a `2026-06`)**. É a versão **fixa e auditada** dos dois períodos do diagnóstico; para outros períodos, a tela usa a `fato_segmento_mensal` (A3).

**Fonte (Gold):** `fato_consumo_faturamento`. As colunas de GD e de troca de medidor também vêm dela (`possui_gd_no_mes`, `troca_medidor`), que garante a **mesma regra de "GD no mês"** das outras tabelas.

**Fórmulas** (V = volume em kWh; P = receita ÷ volume), as do contrato:

| Coluna | Regra |
|---|---|
| `efeito_volume` | `(V1 − V0) × P0` |
| `efeito_preco` | `(P1 − P0) × V1`, calculado como `variacao_receita − efeito_volume` (a mesma conta, e a soma sai **exata**) |
| soma | `efeito_volume + efeito_preco = receita_1 − receita_0` |
| volume zero em um dos períodos | preço do período fica nulo; `efeito_volume = receita_1 − receita_0` e `efeito_preco = 0` |

Receita = `valor_energia` (sem tributos), a definição fechada no contrato.

**Por que a tabela passa por uma tabela temporária (`_stg`):** o Spark serverless não permite `.cache()`, e cada checagem recalcularia a agregação sobre as 2,1 milhões de linhas da fato. A tabela é gravada primeiro em `gold.receita_decomposicao_stg`; as checagens leem dela; **só se todas passarem** a final é gravada e a temporária é apagada.

### ⚠️ Atenção para quem consome: não some os efeitos
`volume_kwh_*` e `receita_*` **são aditivos**: podem ser somados entre linhas. `efeito_preco` e `efeito_volume` **não são**. Cada linha usa o **preço de referência do próprio segmento**, e o preço médio de um conjunto mistura os preços dos segmentos (efeito composição, ou *mix*). Por isso:
- a soma dos efeitos das linhas **não é** o efeito da empresa (a diferença é o efeito composição);
- **para qualquer nível acima da linha** (município, classe, empresa), some `volume_kwh_0/1` e `receita_0/1` das linhas e **aplique as fórmulas de novo**, como a regra para a API da A3 já diz.

A seção 8 do notebook mede essa diferença em cada nível.

**O que entra no dicionário de dados:**
- **bandeira:** o adicional não vem separado na fatura (está embutido em `valor_energia`), então o `efeito_preco` mistura **reajuste de tarifa e bandeira**. A separação é uma estimativa feita no diagnóstico (no residencial, ~13% de aumento contra 10,5% de reajuste);
- **GD:** `volume_kwh_ucs_com_gd_0` e `_1` são o volume, nos dois períodos, das UCs que têm GD conectada no fim do período 1. A diferença é a queda associada à geração distribuída (associação, e não causa comprovada);
- **trocas de medidor:** `qtd_trocas_medidor_1` conta os UC-meses com troca no período 1 (todas as 6.299 estão nele).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from decimal import Decimal
from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_fato = f"{gold_schema}.fato_consumo_faturamento"
tabela_stg = f"{gold_schema}.receita_decomposicao_stg"
tabela_gold = f"{gold_schema}.receita_decomposicao"
tabela_dq_log = f"{silver_schema}.dq_log"      # mesmo log das Silvers e das outras Golds

P1_FIM = "2026-06"                              # fim do período 1: GD "conectada até o fim do período 1"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_fato}")
print(f"destino: {tabela_gold} (via {tabela_stg})")

origem:  v_energy.gold.fato_consumo_faturamento
destino: v_energy.gold.receita_decomposicao (via v_energy.gold.receita_decomposicao_stg)


## 2. Funções de checagem de qualidade

Mesmas funções da aula e das Silvers (`dq_check`, `dq_check_unique` e `dq_check_contagem`), com as mesmas colunas no log (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`). Acrescento `dq_check_valor`, para comparar um número calculado com um esperado (somas que precisam bater com as Silvers e com o diagnóstico); ela grava no log nas mesmas colunas, com `failed_rows` igual a 0 ou 1.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Gold**.

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
            failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura da fato e conferência das colunas

Antes de construir, o notebook confere que a `gold.fato_consumo_faturamento` existe **com as colunas de que precisa**. Se faltar alguma, para aqui com uma mensagem clara.

Só entram as linhas dos dois períodos do diagnóstico (`periodo_diagnostico` igual a `PRIMEIRO_ANO` ou `ULTIMOS_12M`); as 24 mensagens do meio (`OUTRO`) ficam de fora.

Contagens esperadas: a fato tem **2.152.336** linhas, e os dois períodos somam **1.076.035** (536.267 no primeiro ano e 539.768 nos últimos 12 meses).

In [0]:
fato_all = spark.table(tabela_fato)

exigidas = ["id_uc", "mes_referencia", "periodo_diagnostico", "municipio", "bairro", "classe", "possui_gd_no_mes",
            "troca_medidor", "consumo_kwh", "valor_energia", "bandeira_tarifaria"]
faltando = [c for c in exigidas if c not in fato_all.columns]
if faltando:
    raise Exception(f"{tabela_fato} não tem as colunas {faltando} (DESCRIBE {tabela_fato}).")

f = (fato_all.filter(F.col("periodo_diagnostico").isin("PRIMEIRO_ANO", "ULTIMOS_12M"))
     .withColumn("p", F.when(F.col("periodo_diagnostico") == "PRIMEIRO_ANO", 0).otherwise(1)))

print(f"fato: {n_br(fato_all.count())} linhas | nos dois períodos: {n_br(f.count())}")   # esperado: 2.152.336 | 1.076.035
f.groupBy("periodo_diagnostico").count().show()                                           # esperado: 536.267 e 539.768

fato: 2.152.336 linhas | nos dois períodos: 1.076.035
+-------------------+------+
|periodo_diagnostico| count|
+-------------------+------+
|        ULTIMOS_12M|539768|
|       PRIMEIRO_ANO|536267|
+-------------------+------+



## 4. UCs com GD no fim do período 1

**Decisão:** uma UC conta como "com GD" nesta tabela se `possui_gd_no_mes` é verdadeiro em **`2026-06`**, o último mês da base. É a regra "GD conectada até o fim do período 1" do contrato, aplicada com a mesma coluna das outras tabelas (A2, A3 e A5).

**Evidência:** são **3.510 UCs** (todas as conexões da Silver de GD são até 2026-06, verificado na fato). O volume dessas UCs cai de 18,30 milhões de kWh no primeiro ano para 7,10 milhões nos últimos 12 meses (diagnóstico, seção 5.2).

Para cada UC, a marca vale **nos dois períodos** (a UC já "tinha GD" ou não no fim do período 1, e o volume dela no primeiro ano é o de antes do painel): é isso que permite medir a queda associada à GD.

**Conferência:** 3.510 UCs.

In [0]:
gd_fim = (fato_all.filter((F.col("mes_referencia") == P1_FIM) & F.col("possui_gd_no_mes"))
          .select("id_uc").distinct().withColumn("gd_fim_p1", F.lit(True)))
print("UCs com GD no fim do período 1:", n_br(gd_fim.count()))    # esperado: 3.510

f = f.join(F.broadcast(gd_fim), "id_uc", "left").withColumn("gd_fim_p1", F.coalesce("gd_fim_p1", F.lit(False)))

UCs com GD no fim do período 1: 3.510


## 5. Agregação por município × bairro × classe

Uma passada pela fato, com somas condicionais por período:
- **UCs, volume e receita** de cada período (`volume_kwh_*` = soma de `consumo_kwh`; `receita_*` = soma de `valor_energia`);
- **GD:** quantidade de UCs com GD no fim do período 1 e o volume delas nos dois períodos;
- **Trocas de medidor** no período 1 (UC-meses com `troca_medidor`);
- **Faturas não verdes** de cada período, para o percentual de bandeira.

Segmento sem linhas em um período (por exemplo, um bairro que só tem UCs ligadas depois) fica com volume, receita e UCs **zero** nesse período, e não nulo. É o caso de "volume zero" do contrato.

O número de linhas da tabela é o de combinações existentes; **não foi medido** (no máximo, 297 pares município + bairro × 5 classes).

In [0]:
def soma_p(coluna, p, extra=None):
    cond = (F.col("p") == p) if extra is None else ((F.col("p") == p) & extra)
    return F.coalesce(F.sum(F.when(cond, F.col(coluna))), F.lit(0))

agg = f.groupBy("municipio", "bairro", "classe").agg(
    F.countDistinct(F.when(F.col("p") == 0, F.col("id_uc"))).alias("qtd_ucs_periodo_0"),
    F.countDistinct(F.when(F.col("p") == 1, F.col("id_uc"))).alias("qtd_ucs_periodo_1"),
    soma_p("consumo_kwh", 0).alias("volume_kwh_0"),
    soma_p("consumo_kwh", 1).alias("volume_kwh_1"),
    soma_p("valor_energia", 0).alias("receita_0"),
    soma_p("valor_energia", 1).alias("receita_1"),
    F.countDistinct(F.when(F.col("gd_fim_p1"), F.col("id_uc"))).alias("qtd_ucs_com_gd_1"),
    soma_p("consumo_kwh", 0, F.col("gd_fim_p1")).alias("volume_kwh_ucs_com_gd_0"),
    soma_p("consumo_kwh", 1, F.col("gd_fim_p1")).alias("volume_kwh_ucs_com_gd_1"),
    F.count(F.when((F.col("p") == 1) & F.col("troca_medidor"), 1)).alias("qtd_trocas_medidor_1"),
    F.count(F.when(F.col("p") == 0, 1)).alias("faturas_0"),
    F.count(F.when(F.col("p") == 1, 1)).alias("faturas_1"),
    F.count(F.when((F.col("p") == 0) & (F.col("bandeira_tarifaria") != "Verde"), 1)).alias("nao_verde_0"),
    F.count(F.when((F.col("p") == 1) & (F.col("bandeira_tarifaria") != "Verde"), 1)).alias("nao_verde_1"),
)
print("segmentos (município x bairro x classe):", n_br(agg.count()))

segmentos (município x bairro x classe): 1.106


## 6. Preço, variação e os efeitos preço e volume

**Decisão:** aplicar as fórmulas do contrato linha a linha.

- **Preço médio:** `receita ÷ volume`, com 4 casas; nulo se o volume do período é zero.
- **Variação da receita:** `receita_1 − receita_0`; em % sobre `receita_0` (nula se `receita_0` é zero). A variação do volume em % segue o mesmo modelo.
- **`efeito_volume`:** `(V1 − V0) × P0`, se o volume dos dois períodos é maior que zero; senão, **toda a variação** (regra do volume zero).
- **`efeito_preco`:** `variacao_receita − efeito_volume`. É igual a `(P1 − P0) × V1` (se `P1 = R1/V1` e `P0 = R0/V0`, as duas contas dão `R1 − R0·V1/V0`), com a vantagem de a soma sair **exata ao centavo**, sem os dois arredondamentos separados. Quando o volume de um período é zero, resulta em **0**, como o contrato manda.
- **Consumo médio por UC:** `volume ÷ UCs com fatura` do período, com 2 casas.
- **% de faturas não verdes:** das faturas do período, as com bandeira diferente de `Verde`. No primeiro ano é **0%** (100% das faturas eram verdes) e nos últimos 12 meses fica em torno de **67%**.

Os cálculos intermediários usam `double` (15 a 16 dígitos), mais que suficiente para valores de até alguns milhões de reais com 2 casas. O resultado é gravado em `decimal`.

**Cuidado:** a divisão usa `try_divide`, que devolve nulo no lugar de erro quando o divisor é zero (o modo ANSI do serverless interrompe a divisão por zero).

In [0]:
def dbl(c): return F.col(c).cast("double")

df = (
    agg
    .withColumn("volume_kwh_0", F.col("volume_kwh_0").cast("decimal(18,2)"))
    .withColumn("volume_kwh_1", F.col("volume_kwh_1").cast("decimal(18,2)"))
    .withColumn("receita_0", F.col("receita_0").cast("decimal(18,2)"))
    .withColumn("receita_1", F.col("receita_1").cast("decimal(18,2)"))
    .withColumn("preco_medio_0", F.round(F.expr("try_divide(receita_0, volume_kwh_0)"), 4).cast("decimal(12,4)"))
    .withColumn("preco_medio_1", F.round(F.expr("try_divide(receita_1, volume_kwh_1)"), 4).cast("decimal(12,4)"))
    .withColumn("variacao_receita", (F.col("receita_1") - F.col("receita_0")).cast("decimal(18,2)"))
    .withColumn("variacao_receita_pct", F.round(F.expr("try_divide(variacao_receita, receita_0)") * 100, 2).cast("decimal(10,2)"))
    .withColumn("variacao_volume_pct", F.round(F.expr("try_divide(volume_kwh_1 - volume_kwh_0, volume_kwh_0)") * 100, 2).cast("decimal(10,2)"))
    .withColumn(
        "efeito_volume",
        F.when((F.col("volume_kwh_0") > 0) & (F.col("volume_kwh_1") > 0),
               F.round(F.expr("try_divide(cast(volume_kwh_1 - volume_kwh_0 as double) * cast(receita_0 as double), cast(volume_kwh_0 as double))"), 2))
         .otherwise(F.col("variacao_receita").cast("double"))
         .cast("decimal(18,2)"),
    )
    .withColumn("efeito_preco", (F.col("variacao_receita") - F.col("efeito_volume")).cast("decimal(18,2)"))
    .withColumn("consumo_medio_uc_0", F.round(F.expr("try_divide(volume_kwh_0, qtd_ucs_periodo_0)"), 2).cast("decimal(12,2)"))
    .withColumn("consumo_medio_uc_1", F.round(F.expr("try_divide(volume_kwh_1, qtd_ucs_periodo_1)"), 2).cast("decimal(12,2)"))
    .withColumn("volume_kwh_ucs_com_gd_0", F.col("volume_kwh_ucs_com_gd_0").cast("decimal(18,2)"))
    .withColumn("volume_kwh_ucs_com_gd_1", F.col("volume_kwh_ucs_com_gd_1").cast("decimal(18,2)"))
    .withColumn("pct_faturas_bandeira_nao_verde_0", F.round(F.expr("try_divide(nao_verde_0, faturas_0)") * 100, 2).cast("decimal(5,2)"))
    .withColumn("pct_faturas_bandeira_nao_verde_1", F.round(F.expr("try_divide(nao_verde_1, faturas_1)") * 100, 2).cast("decimal(5,2)"))
)

df_rec = df.select(
    "municipio", "bairro", "classe",
    "qtd_ucs_periodo_0", "qtd_ucs_periodo_1",
    "volume_kwh_0", "volume_kwh_1", "receita_0", "receita_1",
    "preco_medio_0", "preco_medio_1",
    "variacao_receita", "variacao_receita_pct", "variacao_volume_pct",
    "efeito_preco", "efeito_volume",
    "consumo_medio_uc_0", "consumo_medio_uc_1",
    "qtd_ucs_com_gd_1", "volume_kwh_ucs_com_gd_0", "volume_kwh_ucs_com_gd_1",
    "qtd_trocas_medidor_1",
    "pct_faturas_bandeira_nao_verde_0", "pct_faturas_bandeira_nao_verde_1",
)
print("colunas:", len(df_rec.columns))    # esperado: 24
df_rec.printSchema()

# Grava na tabela temporária; as checagens leem dela
(
    df_rec.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_stg)
)
rec = spark.table(tabela_stg)
print(f"{tabela_stg}: {n_br(rec.count())} linhas")

colunas: 24
root
 |-- municipio: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- classe: string (nullable = true)
 |-- qtd_ucs_periodo_0: long (nullable = false)
 |-- qtd_ucs_periodo_1: long (nullable = false)
 |-- volume_kwh_0: decimal(18,2) (nullable = true)
 |-- volume_kwh_1: decimal(18,2) (nullable = true)
 |-- receita_0: decimal(18,2) (nullable = true)
 |-- receita_1: decimal(18,2) (nullable = true)
 |-- preco_medio_0: decimal(12,4) (nullable = true)
 |-- preco_medio_1: decimal(12,4) (nullable = true)
 |-- variacao_receita: decimal(18,2) (nullable = true)
 |-- variacao_receita_pct: decimal(10,2) (nullable = true)
 |-- variacao_volume_pct: decimal(10,2) (nullable = true)
 |-- efeito_preco: decimal(18,2) (nullable = true)
 |-- efeito_volume: decimal(18,2) (nullable = true)
 |-- consumo_medio_uc_0: decimal(12,2) (nullable = true)
 |-- consumo_medio_uc_1: decimal(12,2) (nullable = true)
 |-- qtd_ucs_com_gd_1: long (nullable = false)
 |-- volume_kwh_ucs_com_gd_0: de

## 7. Checagens de qualidade

Rodam sobre a tabela temporária, **antes** de gravar a final:

| Checagem | Protege |
|---|---|
| `(municipio, bairro, classe)` único; colunas-chave não nulas; `classe` nas 5 classes | grão da tabela |
| `efeito_volume + efeito_preco = variacao_receita` (até 1 centavo) | a decomposição fecha em cada linha |
| Volume zero: `efeito_preco = 0` e preço do período nulo | a regra do contrato |
| Preço preenchido quando há volume | divisões |
| **Soma dos volumes = 227.052.927 e 218.079.054 kWh** | a tabela reproduz o diagnóstico e a fato |
| **Soma das receitas = R$ 117.830.394,38 e R$ 127.262.595,65** | idem, em reais |
| **UCs: 44.741 e 45.000** | UCs com fatura em cada período |
| **Efeito global preço e volume = +R$ 14.089.243 e −R$ 4.657.042** (até R$ 1) | as fórmulas aplicadas ao total dão os números do diagnóstico |
| **3.510 UCs com GD** e volumes de 18,30 e 7,10 milhões de kWh | a GD do diagnóstico |
| **6.299 trocas de medidor** no período 1 | os UC-meses de troca |
| % de bandeira não verde no primeiro ano = 0 | 100% das faturas eram verdes |

Os valores esperados vêm do `docs/diagnostico_receita.md` e das execuções anteriores (a fato e a `dim_uc`); numa reexecução com dados novos, precisam ser atualizados.

A **soma dos efeitos das linhas** não é checagem: ela é **diferente** do efeito global, por causa do efeito composição, e a seção 8 mede essa diferença.

In [0]:
t = "gold.receita_decomposicao"
dq_results.clear()   # se esta célula for reexecutada, não acumula resultados antigos

CLASSES = ["RESIDENCIAL", "RURAL", "COMERCIAL", "PODER PUBLICO", "INDUSTRIAL"]
n_rec = rec.count()

dq_check_unique(t, "(municipio, bairro, classe) único", rec, ["municipio", "bairro", "classe"])
for coluna in ["municipio", "bairro", "classe", "qtd_ucs_periodo_0", "qtd_ucs_periodo_1", "volume_kwh_0", "volume_kwh_1",
               "receita_0", "receita_1", "variacao_receita", "efeito_preco", "efeito_volume",
               "qtd_ucs_com_gd_1", "volume_kwh_ucs_com_gd_0", "volume_kwh_ucs_com_gd_1", "qtd_trocas_medidor_1"]:
    dq_check(t, f"{coluna} não nulo", rec, F.col(coluna).isNotNull())
dq_check(t, "classe só nas 5 classes", rec, F.coalesce(F.col("classe").isin(*CLASSES), F.lit(False)))

# A decomposição fecha em cada linha, e o caso de volume zero
dq_check(t, "efeito_volume + efeito_preco = variacao_receita (até 1 centavo)", rec,
         F.abs(F.col("efeito_volume") + F.col("efeito_preco") - F.col("variacao_receita")) <= F.lit(Decimal("0.01")))
dq_check(t, "volume zero em um período: efeito_preco = 0 e efeito_volume = variacao_receita", rec,
         ((F.col("volume_kwh_0") > 0) & (F.col("volume_kwh_1") > 0)) | ((F.col("efeito_preco") == 0) & (F.col("efeito_volume") == F.col("variacao_receita"))))
dq_check(t, "preço do período preenchido quando há volume", rec,
         ((F.col("volume_kwh_0") == 0) | F.col("preco_medio_0").isNotNull()) & ((F.col("volume_kwh_1") == 0) | F.col("preco_medio_1").isNotNull()))
dq_check(t, "volumes e receitas >= 0", rec,
         (F.col("volume_kwh_0") >= 0) & (F.col("volume_kwh_1") >= 0) & (F.col("receita_0") >= 0) & (F.col("receita_1") >= 0))
dq_check(t, "bandeira não verde = 0% no primeiro ano", rec,
         F.col("pct_faturas_bandeira_nao_verde_0").isNull() | (F.col("pct_faturas_bandeira_nao_verde_0") == 0))

# Reconciliação com o diagnóstico e com a fato (somas exatas)
tot = rec.agg(
    F.sum("volume_kwh_0").alias("v0"), F.sum("volume_kwh_1").alias("v1"),
    F.sum("receita_0").alias("r0"), F.sum("receita_1").alias("r1"),
    F.sum("qtd_ucs_periodo_0").alias("u0"), F.sum("qtd_ucs_periodo_1").alias("u1"),
    F.sum("qtd_ucs_com_gd_1").alias("gd"), F.sum("qtd_trocas_medidor_1").alias("trocas"),
    F.sum("volume_kwh_ucs_com_gd_0").alias("gd_v0"), F.sum("volume_kwh_ucs_com_gd_1").alias("gd_v1"),
).first()

dq_check_valor(t, "volume do primeiro ano = 227.052.927 kWh", n_rec, tot["v0"], Decimal("227052927.00"))
dq_check_valor(t, "volume dos últimos 12 meses = 218.079.054 kWh", n_rec, tot["v1"], Decimal("218079054.00"))
dq_check_valor(t, "receita do primeiro ano = R$ 117.830.394,38", n_rec, tot["r0"], Decimal("117830394.38"))
dq_check_valor(t, "receita dos últimos 12 meses = R$ 127.262.595,65", n_rec, tot["r1"], Decimal("127262595.65"))
dq_check_valor(t, "UCs com fatura no primeiro ano = 44.741", n_rec, tot["u0"], 44_741)
dq_check_valor(t, "UCs com fatura nos últimos 12 meses = 45.000", n_rec, tot["u1"], 45_000)
dq_check_valor(t, "UCs com GD no fim do período 1 = 3.510", n_rec, tot["gd"], 3_510)
dq_check_valor(t, "trocas de medidor no período 1 = 6.299", n_rec, tot["trocas"], 6_299)
dq_check_valor(t, "volume das UCs com GD no primeiro ano = 18,30 milhões de kWh", n_rec, round(float(tot["gd_v0"]) / 1e6, 2), 18.30)
dq_check_valor(t, "volume das UCs com GD nos últimos 12 meses = 7,10 milhões de kWh", n_rec, round(float(tot["gd_v1"]) / 1e6, 2), 7.10)

# Efeitos no total da empresa: as fórmulas do contrato aplicadas aos totais (diagnóstico, seção 3)
V0, V1, R0, R1 = [float(tot[c]) for c in ("v0", "v1", "r0", "r1")]
ef_vol_global = (V1 - V0) * (R0 / V0)
ef_preco_global = (R1 - R0) - ef_vol_global
print(f"efeito global: volume {ef_vol_global:,.2f} | preço {ef_preco_global:,.2f} | variação {R1 - R0:,.2f}")
dq_check_valor(t, "efeito-volume global = -R$ 4.657.042 (até R$ 1)", n_rec, abs(ef_vol_global - (-4_657_042)) <= 1, True)
dq_check_valor(t, "efeito-preço global = +R$ 14.089.243 (até R$ 1)", n_rec, abs(ef_preco_global - 14_089_243) <= 1, True)

# Informativo: % de faturas não verdes nos últimos 12 meses (contrato: ~67%; diagnóstico: 66,8%)
nv = f.agg((F.sum(F.when(F.col("p") == 1, (F.col("bandeira_tarifaria") != "Verde").cast("int"))) * 100
            / F.count(F.when(F.col("p") == 1, 1))).alias("pct")).first()["pct"]
print(f"faturas não verdes nos últimos 12 meses: {nv:.1f}%")    # esperado: ~66,8%

[PASS] gold.receita_decomposicao | (municipio, bairro, classe) único | 0 chaves duplicadas de 1106 linhas
[PASS] gold.receita_decomposicao | municipio não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | bairro não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | classe não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | qtd_ucs_periodo_0 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | qtd_ucs_periodo_1 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | volume_kwh_0 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | volume_kwh_1 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | receita_0 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | receita_1 não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | variacao_receita não nulo | 0/1106 linhas falharam
[PASS] gold.receita_decomposicao | efeito_preco não nulo | 0/1106 linhas falharam

## 8. O efeito composição: por que a soma das linhas não é o efeito da empresa

Aplica as fórmulas do contrato **em cada nível** (empresa, classe, município e linha) e soma os efeitos dos grupos de cada nível. A receita total que varia é a mesma em todos os níveis; o que muda é a **divisão** entre preço e volume. A diferença para o efeito da empresa é o **efeito composição**.

Esperado: no nível da empresa, `efeito_volume = −R$ 4.657.042` e `efeito_preco = +R$ 14.089.243`; no nível de classe, a soma dos efeitos-volume do diagnóstico (seção 5.1) é de aproximadamente **−R$ 5,0 milhões** (as linhas da tabela somam −5,00 e o texto fala em −4,99, por arredondamento). Se os dois números do diagnóstico estiverem no mesmo texto, é preciso explicar a diferença (veja a observação na descrição do PR).

In [0]:
def efeitos_no_nivel(nome, chaves):
    base = rec.groupBy(*chaves) if chaves else rec.groupBy()
    g = base.agg(F.sum("volume_kwh_0").alias("v0"), F.sum("volume_kwh_1").alias("v1"),
                 F.sum("receita_0").alias("r0"), F.sum("receita_1").alias("r1"))
    g = g.withColumn("ev", F.when((F.col("v0") > 0) & (F.col("v1") > 0),
                                  (F.col("v1") - F.col("v0")).cast("double") * F.col("r0").cast("double") / F.col("v0").cast("double"))
                            .otherwise((F.col("r1") - F.col("r0")).cast("double")))
    g = g.withColumn("ep", (F.col("r1") - F.col("r0")).cast("double") - F.col("ev"))
    return g.agg(F.lit(nome).alias("nivel"), F.count("*").alias("grupos"),
                 F.round(F.sum("ev"), 0).alias("efeito_volume"), F.round(F.sum("ep"), 0).alias("efeito_preco"),
                 F.round(F.sum(F.col("r1") - F.col("r0")).cast("double"), 0).alias("variacao_receita"))

niveis = (efeitos_no_nivel("1. empresa", [])
          .union(efeitos_no_nivel("2. classe", ["classe"]))
          .union(efeitos_no_nivel("3. município", ["municipio"]))
          .union(efeitos_no_nivel("4. município x classe", ["municipio", "classe"]))
          .union(efeitos_no_nivel("5. linha (município x bairro x classe)", ["municipio", "bairro", "classe"])))
display(niveis.orderBy("nivel"))
# a variação da receita é a mesma nos 5 níveis (~R$ 9.432.201); a divisão preço x volume muda de nível para nível

nivel,grupos,efeito_volume,efeito_preco,variacao_receita
1. empresa,1,-4657042.0,1.4089243E7,9432201.0
2. classe,5,-4994045.0,1.4426246E7,9432201.0
3. município,14,-4662467.0,1.4094669E7,9432201.0
4. município x classe,70,-4994034.0,1.4426235E7,9432201.0
5. linha (município x bairro x classe),1106,-4993510.0,1.4425712E7,9432201.0


## 9. Gravação da Gold, só se todas as checagens passarem

Se **qualquer** checagem falhar, a tabela final **não é gravada**: o notebook imprime quais falharam e é interrompido com erro. A `_stg` fica para inspeção, e a `receita_decomposicao` anterior, se existir, permanece como estava. Se todas passarem, a final é gravada a partir da `_stg`, e a `_stg` é apagada.

In [0]:
# Persistência do dq_log (modo append); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada ({tabela_stg} ficou para inspeção):")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    rec.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold)
)
spark.sql(f"DROP TABLE IF EXISTS {tabela_stg}")
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

Todas as 35 checagens passaram. v_energy.gold.receita_decomposicao gravada: 1.106 linhas


## 10. Teste de conexão e respostas ao Henrique

1. **Conexão com a `dim_uc`:** todo par município + bairro e toda classe da decomposição existem na `gold.dim_uc`.
2. **"Qual bairro mais perdeu volume residencial?"** (a pergunta do Henrique, que o diagnóstico deixou em aberto): os 15 piores bairros da classe residencial, e quanto da perda de cada um vem das UCs com GD e quanto vem do resto.
3. **Por município:** a variação do volume e da receita de cada município, com os efeitos recalculados **no nível do município** (somando volumes e receitas e aplicando as fórmulas de novo).
4. **A queda associada à GD:** o volume das UCs com GD nos dois períodos, por classe.

In [0]:
r = spark.table(tabela_gold)

# 1. Conexão com a dim_uc (só roda se a gold.dim_uc existir neste workspace)
try:
    dim = spark.table(f"{gold_schema}.dim_uc").select("municipio", "bairro", "classe").distinct()
    print("segmentos sem correspondência na gold.dim_uc:",
          n_br(r.select("municipio", "bairro", "classe").join(dim, ["municipio", "bairro", "classe"], "left_anti").count()))   # esperado: 0
except Exception:
    print("gold.dim_uc não existe neste workspace: a conexão fica para quem tiver as duas tabelas")

# 2. Os 15 bairros que mais perderam volume residencial (volume em kWh; efeito-volume em R$)
display(
    r.filter("classe = 'RESIDENCIAL'")
    .withColumn("perda_kwh", F.col("volume_kwh_1") - F.col("volume_kwh_0"))
    .select("municipio", "bairro", "qtd_ucs_periodo_1", "volume_kwh_0", "volume_kwh_1", "perda_kwh",
            "variacao_volume_pct", "efeito_volume", "qtd_ucs_com_gd_1")
    .orderBy("perda_kwh").limit(15)
)

segmentos sem correspondência na gold.dim_uc: 0


municipio,bairro,qtd_ucs_periodo_1,volume_kwh_0,volume_kwh_1,perda_kwh,variacao_volume_pct,efeito_volume,qtd_ucs_com_gd_1
Recife,Iputinga,1053,2242415.00,1746316.00,-496099.00,-22.12,-265838.23,97
Jaboatão dos Guararapes,Jardim Jordão,946,2020315.00,1580984.00,-439331.00,-21.75,-235416.70,76
Recife,Macaxeira,476,1061271.00,828593.00,-232678.00,-21.92,-124682.39,32
Jaboatão dos Guararapes,Prazeres,550,1161870.00,948658.00,-213212.00,-18.35,-114248.87,42
Recife,Brasília Teimosa,1797,3827611.00,3666105.00,-161506.00,-4.22,-86546.08,146
Jaboatão dos Guararapes,Barra de Jangada,586,1264620.00,1187876.00,-76744.00,-6.07,-41122.93,59
São Lourenço da Mata,Pixete,605,1291538.00,1223105.00,-68433.00,-5.30,-36670.34,55
Paulista,Fragoso,759,1644950.00,1577461.00,-67489.00,-4.10,-36163.60,63
Igarassu,Saramandaia,410,868658.00,805863.00,-62795.00,-7.23,-33648.96,46
Jaboatão dos Guararapes,Sucupira,531,1135215.00,1073879.00,-61336.00,-5.40,-32866.98,41


In [0]:
# 2b. Nesses bairros, quanto da perda vem das UCs com GD e quanto vem do resto?
# pct_perda_sem_gd = variação do volume das UCs SEM GD (sobre o volume delas no primeiro ano)
res = (
    r.filter("classe = 'RESIDENCIAL'")
    .withColumn("perda_kwh", F.col("volume_kwh_1") - F.col("volume_kwh_0"))
    .withColumn("perda_ucs_gd", F.col("volume_kwh_ucs_com_gd_1") - F.col("volume_kwh_ucs_com_gd_0"))
    .withColumn("perda_sem_gd", F.col("perda_kwh") - F.col("perda_ucs_gd"))
    .withColumn("pct_perda_sem_gd", F.round(F.expr("try_divide(perda_sem_gd, volume_kwh_0 - volume_kwh_ucs_com_gd_0)") * 100, 1))
    .withColumn("pct_ucs_com_gd", F.round(F.expr("try_divide(qtd_ucs_com_gd_1, qtd_ucs_periodo_1)") * 100, 1))
)
display(res.select("municipio", "bairro", "perda_kwh", "perda_ucs_gd", "perda_sem_gd", "pct_perda_sem_gd",
                   "pct_ucs_com_gd", "qtd_trocas_medidor_1").orderBy("perda_kwh").limit(15))

municipio,bairro,perda_kwh,perda_ucs_gd,perda_sem_gd,pct_perda_sem_gd,pct_ucs_com_gd,qtd_trocas_medidor_1
Recife,Iputinga,-496099.00,-197649.00,-298450.00,-15.6,9.2,113
Jaboatão dos Guararapes,Jardim Jordão,-439331.00,-163415.00,-275916.00,-15.7,8.0,89
Recife,Macaxeira,-232678.00,-69401.00,-163277.00,-17.3,6.7,54
Jaboatão dos Guararapes,Prazeres,-213212.00,-86170.00,-127042.00,-12.4,7.6,71
Recife,Brasília Teimosa,-161506.00,-298948.00,137442.00,4.1,8.1,249
Jaboatão dos Guararapes,Barra de Jangada,-76744.00,-124033.00,47289.00,4.5,10.1,84
São Lourenço da Mata,Pixete,-68433.00,-117515.00,49082.00,4.5,9.1,82
Paulista,Fragoso,-67489.00,-133017.00,65528.00,4.6,8.3,93
Igarassu,Saramandaia,-62795.00,-89014.00,26219.00,3.6,11.2,70
Jaboatão dos Guararapes,Sucupira,-61336.00,-101887.00,40551.00,4.1,7.7,75


In [0]:
# 3. Por município (efeitos recalculados no nível do município)
mun = (r.groupBy("municipio").agg(F.sum("volume_kwh_0").alias("v0"), F.sum("volume_kwh_1").alias("v1"),
                                  F.sum("receita_0").alias("r0"), F.sum("receita_1").alias("r1"),
                                  F.sum("qtd_ucs_com_gd_1").alias("ucs_gd"))
       .withColumn("variacao_volume_pct", F.round((F.col("v1") / F.col("v0") - 1) * 100, 1))
       .withColumn("variacao_receita_pct", F.round((F.col("r1") / F.col("r0") - 1) * 100, 1))
       .withColumn("efeito_volume", F.round((F.col("v1") - F.col("v0")).cast("double") * F.col("r0").cast("double") / F.col("v0").cast("double"), 0))
       .withColumn("efeito_preco", F.round((F.col("r1") - F.col("r0")).cast("double") - F.col("efeito_volume"), 0)))
display(mun.select("municipio", "variacao_volume_pct", "variacao_receita_pct", "efeito_volume", "efeito_preco", "ucs_gd")
        .orderBy("efeito_volume"))

# 4. A queda associada à GD, por classe (esperado: ~-61% no residencial e ~-61% no comercial; industrial, rural e poder público não têm GD)
display(
    r.groupBy("classe")
    .agg(F.sum("qtd_ucs_com_gd_1").alias("ucs_com_gd"),
         F.sum("volume_kwh_ucs_com_gd_0").alias("volume_gd_0"), F.sum("volume_kwh_ucs_com_gd_1").alias("volume_gd_1"))
    .withColumn("variacao_pct", F.round((F.expr("try_divide(volume_gd_1, volume_gd_0)") - 1) * 100, 1))   # classes sem GD: nulo, e não erro
    .orderBy("classe")
)

municipio,variacao_volume_pct,variacao_receita_pct,efeito_volume,efeito_preco,ucs_gd
Recife,-4.3,7.7,-2040492.0,5643415.0,1420
Jaboatão dos Guararapes,-6.8,4.8,-1343605.0,2288611.0,569
Paulista,-2.6,9.5,-267602.0,1234889.0,304
Cabo de Santo Agostinho,-3.2,8.8,-228125.0,855133.0,218
Igarassu,-4.1,8.4,-135078.0,411063.0,127
Olinda,-1.2,10.7,-133723.0,1342198.0,328
Camaragibe,-2.4,9.7,-118217.0,600529.0,127
Ipojuca,-2.7,9.8,-81778.0,378880.0,96
São Lourenço da Mata,-2.0,10.1,-74941.0,457399.0,114
Moreno,-3.2,8.0,-67689.0,235926.0,52


classe,ucs_com_gd,volume_gd_0,volume_gd_1,variacao_pct
COMERCIAL,383,7714129.00,3013928.00,-60.9
INDUSTRIAL,0,0.00,0.00,null
PODER PUBLICO,0,0.00,0.00,null
RESIDENCIAL,3127,10584372.00,4087182.00,-61.4
RURAL,0,0.00,0.00,null
